# 2. FranceCrops: small candidate batches and annual TIFFs

Retain the existing source-label, geometry, identity and observation-year checks. Select up to 100 candidate parcels per available crop for an overnight round before preparing masks and requesting imagery. Upstream reserved splits remain excluded from development sampling. Full reference discovery is separate from this bounded satellite collection.

Each selected tile retains all 24 half-month periods, so four-month windows can change later. After the candidate batches complete, notebook 04 applies the existing timing/quality checks and reports usable pixels and parcels by locked train/validation/test partition. Subsequent batches fill only measured deficits.

Annual TIFFs are organized by source, year, campaign and round. Each contains 168 labelled bands; sampling masks are separate. Opening the notebook starts no work. Restart Kernel → Run All prepares/rebuilds masks and submits or collects the current small batch. Batch output reports saved data and cloud job states; completion percentages do not represent elapsed processing time.


## 1. Fresh-clone setup

Choose a Python 3.12+ kernel (3.13 recommended), then **Run All**. Missing packages and local settings are prepared automatically. Enter your Earth Engine project ID once if prompted; existing settings are reused. Collection and sign-in are enabled.

**Output:** the source-table and local archive paths. No pre-existing FranceCrops data is required: the next stage fetches missing reference tables.


In [1]:
# Standard-library setup runs before scientific imports, so a fresh clone needs no saved data.
from pathlib import Path
import sys

notebook_repository_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
notebook_source_folder = str(notebook_repository_root / "src")
if notebook_source_folder not in sys.path:
    sys.path.insert(0, notebook_source_folder)
from crop_pipeline import notebook_setup
notebook_setup.prepare_environment(notebook_repository_root)

from pathlib import Path
import json
import os
from crop_pipeline.sources import francecrops as francecrops_workflow
from crop_pipeline.windows.seasons import file_sha256

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
settings_path = Path("settings.local.json")
local_settings = notebook_setup.ensure_local_settings(settings_path)
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()
revision = francecrops_workflow.DEFAULT_REVISION
source_table_folder = report_folder.parent / "francecrops/source_metadata" / revision
from crop_pipeline.sources import annual

from crop_pipeline.sources import adaptive
run_name = adaptive.COLLECTION_NAME
candidate_parcels_per_crop = 100  # Overnight round: up to 100 cabbage + 100 potato parcels
# Run All starts collection; existing verified files are reused in this versioned archive.
download_missing_reference_tables = True
run_satellite_collection = True
# Small request groups reduce overhead without changing the imagery recipe.
collection_backend = "batch"  # Earth Engine assets; rerun to collect finished jobs
archive_format = "pixel_tables"  # Annual selected-pixel Parquet; raster archives remain available
tile_start = 0
tile_count = None  # Adaptive candidates are already bounded by parcel budgets
download_workers = 2
intervals_per_request = 6
authenticate_now = True
earth_engine_project = os.environ.get("EE_PROJECT", local_settings.get("earth_engine_project", ""))
annual_archive_root = annual.archive_root(local_settings, report_folder)
print("Collection:", run_name)
print("Archive root:", annual_archive_root)
print("Satellite collection enabled:", run_satellite_collection)


# Resolve account setup before large reference transfers start.
earth_engine_project = notebook_setup.project_for_collection(local_settings, settings_path, required=run_satellite_collection)

from crop_pipeline import progress as pipeline_progress
notebook_progress = pipeline_progress.start_notebook(
    "02_francecrops_preparation",
    ['Obtain reference tables', 'Prepare eligible parcels', 'Certificates', 'Collect annual imagery'],
    report_folder.parent / "progress",
    workload={"folder": str(annual_archive_root), "satellite_collection": globals().get("run_satellite_collection"), "training_mode": globals().get("training_mode")},
)


/Users/josedavid/Documents/programming/external-crop-data-classification/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook environment ready: /Users/josedavid/Documents/programming/external-crop-data-classification/.venv/bin/python
Collection: full_year_all_parcels
Archive root: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_archive
Satellite collection enabled: True


02_francecrops_preparation: stages:   0%|          | 0/4 [00:01<?, ?stage/s, elapsed 0m 01s; notebook ETA learning (future stages not yet timed)]

## 2. Obtain the reference tables automatically

Fetch only missing class-map, label and footprint Parquet tables from the pinned FranceCrops release. Reuse existing files without overwriting them. Original interpolated image arrays are not needed. Certificate setup runs before these requests.

**Output:** three local table paths. For an offline inspection, set `download_missing_reference_tables = False`; missing inputs then produce a clear error.


In [2]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Obtain reference tables')

table_paths = francecrops_workflow.download_reference_tables(
    source_table_folder, revision=revision,
    download_if_missing=download_missing_reference_tables,
)
print("Verified reference paths:", table_paths)

pipeline_progress.end_stage("completed")


02_francecrops_preparation: stages:  25%|██▌       | 1/4 [00:00<00:00, 4696.87stage/s, elapsed 0m 01s; notebook ETA learning (future stages not yet timed)]        

Verified reference paths: {'class_map': PosixPath('/Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/francecrops/source_metadata/6b6c51b780d8579138593f92738ed52f2528096e/data/protocol/class_map.parquet'), 'metadata': PosixPath('/Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/francecrops/source_metadata/6b6c51b780d8579138593f92738ed52f2528096e/data/protocol/metadata.parquet'), 'geolocation': PosixPath('/Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/francecrops/source_metadata/6b6c51b780d8579138593f92738ed52f2528096e/data/geolocation/geolocation.parquet')}


## Prepare a bounded candidate parcel batch

Choose a reproducible crop-balanced subset, spread across geographic/year strata, before rasterization. Keep the existing inward buffer and screen contested locations against all reference parcels, including unselected neighbors. Only selected parcels receive masks and satellite requests. Printed estimates describe this candidate batch, not the full reference catalogue.


In [3]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Prepare eligible parcels')

reference_features, reference_audit = francecrops_workflow.load_target_parcels(table_paths, revision, return_audit=True)
adaptive_batches = adaptive.prepare_round_batches(
    "francecrops", reference_features, annual_archive_root, report_folder,
    provenance={"dataset_revision": revision, "tables": {name: file_sha256(path) for name, path in table_paths.items()},
                "unavailable_target_classes": ["carrot"], "upstream_splits_preserved": True, "reference_audit": reference_audit},
    inward_buffer_metres=10, parcels_per_crop=candidate_parcels_per_crop,
)
annual_plan, adaptive_context = next(((plan, context) for plan, context in adaptive_batches if plan is not None), (None, adaptive_batches[-1][1]))
print("Batch parcels to resume/submit:", sum(context["candidate_parcels"] for plan, context in adaptive_batches if plan is not None))
print("Estimate for first active batch:", annual_plan["estimate"] if annual_plan else "No new candidates required")
print("Pixels by year:", {year: inputs["pixels_per_crop"] for year, inputs in annual_plan["years"].items()} if annual_plan else {})

pipeline_progress.set_workload({"folder": str(annual_archive_root), "source_revision": annual_plan["revision"] if annual_plan else "targets_met_or_exhausted", "satellite_collection": run_satellite_collection})

pipeline_progress.end_stage("completed")


02_francecrops_preparation: stages:  50%|█████     | 2/4 [00:03<00:03,  1.69s/stage, elapsed 0m 05s; notebook ETA learning (future stages not yet timed)]                   

Estimate: {'parcels': 13166, 'tiles': 18012, 'composites': 432288, 'raw_raster_bytes': 8539437984, 'raw_raster_gib': 7.953, 'recommended_free_bytes': 17078875968, 'available_bytes': 57898991616, 'note': 'Uncompressed float32 rasters; allow additional disk for summaries, features and models. Compression varies.'}
Pixels by year: {'2022': {'cabbage': 832637, 'potato': 3507840}}


## Collect the full calendar year — enabled for Run All

`run_satellite_collection = True` and `authenticate_now = True` are set for the next Run All. To prepare only, set both to `False`. Earth Engine applies the existing cloud masks, bilinear B5/B11 resampling, daily mosaics and half-month medians. It also calculates dated NDVI, NDRE and NDMI summaries. GeoTIFFs and summary tables are saved locally, with grouped requests and verified-file reuse on restart.

**Output: `status: disabled`** means no authentication, image queries, tasks or downloads ran. **`completed`** means every planned tile has all 24 saved intervals and daily-summary checkpoints. A cloudy interval can be saved successfully and still fail the window notebook's quality checks.


In [4]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Certificates')

import ssl
import certifi

# Use the system's explicit trust setting when one exists; otherwise use certifi.
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
print("Trusted certificates:", ssl.create_default_context().cert_store_stats()["x509_ca"])

pipeline_progress.end_stage("completed")


02_francecrops_preparation: stages:  75%|███████▌  | 3/4 [00:03<00:01,  1.13s/stage, elapsed 0m 05s; notebook ETA learning (future stages not yet timed)]      

Trusted certificates: 121


### Collect a bounded candidate batch

This notebook targets up to 100 cabbage and 100 potato candidate parcels per round, spread across geographic/year strata, before rasterization. Annual histories remain available for changing the four-month window. Re-run the same source notebook to collect its existing jobs. After the candidate batches finish, run notebook 04 to count complete usable windows and write deficits. The next run selects new parcels only for deficient crop/partition combinations. Targets are 1,200 training pixels from at least 60 parcels, 400 validation pixels from at least 20 parcels, and 400 reserved-test pixels from at least 20 parcels per crop, with at most 20 retained pixels per parcel. These are initial sampling targets, not a guarantee of model accuracy.

Completed output is organized under `sen4agrinet_work/annual_tifs/<source>/<year>/<campaign>/round_<number>/`. One TIFF per tile contains 168 labelled bands (24 periods × seven measurements). Surrounding unselected pixels are nodata. The compact Parquet files remain the export transport/cache; window analysis reads the organized annual TIFFs. Sampling masks are separate and regenerated during source preparation if removed.

No 5,000-tile or full-catalogue download is the default. Source batches and earlier completed rounds remain separate and are combined by notebook 04. Upstream reserved splits are excluded from development sampling, and locked parcel assignments do not change when new batches are added.

**Overnight setting:** `candidate_parcels_per_crop = 100`. Existing candidate parcels count toward that limit. A separate extension preserves already submitted jobs; with 25 existing parcels per crop it adds at most 75 new ones per crop. Rerunning resumes both batches. Wait until Run All returns normally before shutting down. Cloud jobs continue while the laptop is off; local collection and annual TIFF creation happen on a later run. Usable-pixel targets and locked splits are unchanged.


In [5]:
from crop_pipeline import progress as pipeline_progress
pipeline_progress.begin_stage('Collect annual imagery')

collection_result = adaptive.collect_batches(
    adaptive_batches, earth_engine_project,
    enabled=run_satellite_collection, authenticate=authenticate_now,
    tile_start=tile_start, tile_count=tile_count,
    download_workers=download_workers, intervals_per_request=intervals_per_request,
)
if collection_result.get("progress_by_year"):
    print({key: collection_result[key] for key in ("status", "selected_batches", "registry") if key in collection_result})
else:
    print(collection_result)

pipeline_progress.end_stage("skipped" if not run_satellite_collection else "completed" if collection_result["status"] == "completed" else "pending")
pipeline_progress.finish_notebook()


02_francecrops_preparation: Annual imagery tiles attempted:   0%|          | 0/18012 [00:58<?, ?tile/s, elapsed 1m 04s; notebook ETA learning (future stages not yet timed)]

KeyboardInterrupt: 

02_francecrops_preparation (stopped; fix the error and rerun):   0%|          | 0/18012 [00:59<?, ?tile/s, elapsed 1m 04s; notebook ETA learning (future stages not yet timed)]

02_francecrops_preparation: Annual imagery tiles attempted:   0%|          | 0/18012 [01:03<?, ?tile/s, elapsed 1m 09s; notebook ETA learning (future stages not yet timed)]   

## Assess usable samples in notebook 04

Finish any launched candidate batches, then run [04_four_month_windows.ipynb](04_four_month_windows.ipynb). It combines completed rounds, applies the unchanged timing and quality checks, and retains at most 20 usable pixels per parcel. If crop/partition targets are short, run source notebooks again to select new candidates for those deficits. Continue to 05–06 only after targets are met.

Changing timing rules can reuse annual histories. A cycle crossing a reference-year boundary remains excluded unless adjacent-year imagery is explicitly available. Source labels and reserved evaluation assignments are not replaced to fill a quota.
